# Getting Familiar: Planar Quadrotor (2D Flight Dynamics)

This notebook is the companion to `docs/guides/getting_familiar/06_planar_quadrotor.md`. It provides hands-on exploration of the planar quadrotor model.

**Learning objectives:**
- Understand the 2D quadrotor state and control vectors
- Calculate and verify hover equilibrium
- Analyze thrust response and differential thrust
- Explore energy and power relationships

**Topics covered:**
1. System overview (state, control, dynamics)
2. Hover equilibrium and stability
3. Thrust response analysis
4. Energy and power plots

---
## Setup and Imports

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt

import jax
import jax.numpy as jnp

from fmd.simulator import (
    PlanarQuadrotor,
    simulate,
    ConstantControl,
    ZeroControl,
    PQUAD_X,
    PQUAD_Z,
    PQUAD_THETA,
    PQUAD_X_DOT,
    PQUAD_Z_DOT,
    PQUAD_THETA_DOT,
    PQUAD_T1,
    PQUAD_T2,
)
from fmd.simulator.params import (
    PlanarQuadrotorParams,
    PLANAR_QUAD_TEST_DEFAULT,
)

print(f"JAX backend: {jax.default_backend()}")

---
## 1. System Overview

The planar quadrotor is a 6-state, 2-control system that models a quadrotor constrained to the x-z (vertical) plane.

### State Vector (6 elements)

| Index | Symbol | Description | Units |
|-------|--------|-------------|-------|
| 0 | x | Horizontal position | m |
| 1 | z | Vertical position (positive = up) | m |
| 2 | theta | Pitch angle (positive = nose up) | rad |
| 3 | x_dot | Horizontal velocity | m/s |
| 4 | z_dot | Vertical velocity | m/s |
| 5 | theta_dot | Pitch rate | rad/s |

### Control Vector (2 elements)

| Index | Symbol | Description | Units |
|-------|--------|-------------|-------|
| 0 | T1 | Right rotor thrust | N |
| 1 | T2 | Left rotor thrust | N |

### Dynamics Equations

$$T = T_1 + T_2 \quad \text{(total thrust)}$$
$$M = (T_1 - T_2) \cdot L \quad \text{(pitch moment)}$$

$$\ddot{x} = -\frac{T}{m} \sin(\theta)$$
$$\ddot{z} = \frac{T}{m} \cos(\theta) - g$$
$$\ddot{\theta} = \frac{M}{I}$$

In [ ]:
# Create a planar quadrotor with the test default parameters
quad = PlanarQuadrotor(PLANAR_QUAD_TEST_DEFAULT)

print("Planar Quadrotor Parameters:")
print(f"  Mass: {quad.mass} kg")
print(f"  Arm length: {quad.arm_length} m")
print(f"  Pitch inertia: {quad.inertia_pitch} kg*m^2")
print(f"  Gravity: {quad.g} m/s^2")

print(f"\nState vector size: {quad.num_states}")
print(f"Control vector size: {quad.num_controls}")
print(f"State names: {quad.state_names}")
print(f"Control names: {quad.control_names}")

---
## 2. Hover Equilibrium

At hover, the quadrotor maintains a fixed position with zero velocity and zero pitch. This requires:

1. **Zero net vertical force:** Total thrust equals weight ($T = mg$)
2. **Zero net moment:** Equal thrust from both rotors ($T_1 = T_2$)
3. **Level attitude:** Pitch angle is zero ($\theta = 0$)

The hover control is:
$$T_1 = T_2 = \frac{mg}{2}$$

In [ ]:
# Calculate hover thrust
T_hover_per_rotor = quad.hover_thrust_per_rotor()
T_hover_total = quad.hover_thrust_total()
weight = quad.mass * quad.g

print("Hover Thrust Calculation:")
print(f"  Thrust per rotor: T1 = T2 = {T_hover_per_rotor:.4f} N")
print(f"  Total thrust: T = {T_hover_total:.4f} N")
print(f"  Vehicle weight: mg = {weight:.4f} N")
print(f"  Thrust equals weight? {np.isclose(T_hover_total, weight)}")

In [ ]:
# Verify that hover is an equilibrium (state derivative is zero)
state_hover = quad.default_state()  # All zeros
control_hover = quad.hover_control()  # [T_hover, T_hover]

print(f"Hover state: {np.array(state_hover)}")
print(f"Hover control: {np.array(control_hover)}")

# Compute state derivative at hover
deriv_at_hover = quad.forward_dynamics(state_hover, control_hover)

print(f"\nState derivative at hover:")
print(f"  d/dt[x, z, theta, x_dot, z_dot, theta_dot] = {np.array(deriv_at_hover)}")
print(f"  All derivatives zero? {np.allclose(np.array(deriv_at_hover), 0, atol=1e-12)}")

In [ ]:
# Simulate at hover to verify the quadrotor maintains position
duration = 5.0  # seconds
dt = 0.001  # 1 kHz simulation

result_hover = simulate(
    quad,
    state_hover,
    dt=dt,
    duration=duration,
    control=ConstantControl(control_hover)
)

times = np.array(result_hover.times)
states = np.array(result_hover.states)

print(f"Hover Simulation Results ({duration}s):")
print(f"  Final position: x={states[-1, PQUAD_X]:.2e} m, z={states[-1, PQUAD_Z]:.2e} m")
print(f"  Max position drift: x={np.max(np.abs(states[:, PQUAD_X])):.2e} m, z={np.max(np.abs(states[:, PQUAD_Z])):.2e} m")
print(f"  Max pitch angle: {np.max(np.abs(states[:, PQUAD_THETA])):.2e} rad")

### Hover Stability

Hover is an **unstable equilibrium**. Any perturbation (e.g., initial tilt) will cause the vehicle to drift without active control.

Let's see what happens when we start with a small pitch angle but apply hover thrust.

In [ ]:
# Start with a small perturbation (5 degrees pitch)
theta_perturb = np.radians(5)  # 5 degrees in radians
state_perturbed = quad.create_state(theta=theta_perturb)

# Apply hover control (not corrected for the tilt)
result_perturbed = simulate(
    quad,
    state_perturbed,
    dt=dt,
    duration=3.0,
    control=ConstantControl(control_hover)
)

times_p = np.array(result_perturbed.times)
states_p = np.array(result_perturbed.states)

# Plot the instability
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Position (x-z trajectory)
axes[0].plot(states_p[:, PQUAD_X], states_p[:, PQUAD_Z], 'b-', linewidth=2)
axes[0].plot(states_p[0, PQUAD_X], states_p[0, PQUAD_Z], 'go', markersize=10, label='Start')
axes[0].plot(states_p[-1, PQUAD_X], states_p[-1, PQUAD_Z], 'ro', markersize=10, label='End')
axes[0].set_xlabel('x (m)')
axes[0].set_ylabel('z (m)')
axes[0].set_title('Trajectory with 5 deg Initial Pitch')
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].axis('equal')

# Position vs time
axes[1].plot(times_p, states_p[:, PQUAD_X], 'b-', linewidth=2, label='x')
axes[1].plot(times_p, states_p[:, PQUAD_Z], 'r-', linewidth=2, label='z')
axes[1].set_xlabel('Time (s)')
axes[1].set_ylabel('Position (m)')
axes[1].set_title('Position Drift Over Time')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Pitch angle (constant since no moment)
axes[2].plot(times_p, np.degrees(states_p[:, PQUAD_THETA]), 'b-', linewidth=2)
axes[2].axhline(5, color='r', linestyle='--', alpha=0.7, label='Initial 5 deg')
axes[2].set_xlabel('Time (s)')
axes[2].set_ylabel('Pitch (deg)')
axes[2].set_title('Pitch Angle (Constant)')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"After 3 seconds with 5 deg initial tilt:")
print(f"  x drift: {states_p[-1, PQUAD_X]:.3f} m")
print(f"  z drift: {states_p[-1, PQUAD_Z]:.3f} m")
print(f"  Pitch stays constant at {np.degrees(states_p[-1, PQUAD_THETA]):.2f} deg (no corrective moment)")

---
## 3. Thrust Response Analysis

### 3.1 Vertical Response (Symmetric Thrust)

When both rotors apply equal thrust:
- Excess thrust ($T > mg$) causes climb
- Deficit thrust ($T < mg$) causes descent
- Zero thrust causes freefall

The vertical acceleration is:
$$\ddot{z} = \frac{T}{m} - g \quad \text{(when } \theta = 0 \text{)}$$

In [ ]:
# Compare different thrust levels
thrust_ratios = [0.0, 0.5, 1.0, 1.2, 1.5]
colors = ['red', 'orange', 'black', 'green', 'blue']
labels = ['0% (freefall)', '50%', '100% (hover)', '120%', '150%']

T_hover = quad.hover_thrust_per_rotor()
duration_thrust = 2.0

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ratio, color, label in zip(thrust_ratios, colors, labels):
    T = ratio * T_hover
    control = ConstantControl(jnp.array([T, T]))
    
    result = simulate(quad, quad.default_state(), dt=dt, duration=duration_thrust, control=control)
    times = np.array(result.times)
    states = np.array(result.states)
    
    # Height
    axes[0].plot(times, states[:, PQUAD_Z], color=color, linewidth=2, label=label)
    
    # Vertical velocity
    axes[1].plot(times, states[:, PQUAD_Z_DOT], color=color, linewidth=2, label=label)
    
    # Expected velocity from constant acceleration
    a_expected = (2 * T / quad.mass) - quad.g
    v_expected = a_expected * times
    axes[2].plot(times, v_expected, color=color, linestyle='--', alpha=0.5)
    axes[2].plot(times, states[:, PQUAD_Z_DOT], color=color, linewidth=2, label=label)

axes[0].axhline(0, color='k', linestyle='-', alpha=0.3)
axes[0].set_xlabel('Time (s)')
axes[0].set_ylabel('Height z (m)')
axes[0].set_title('Height vs Time')
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.3)

axes[1].axhline(0, color='k', linestyle='-', alpha=0.3)
axes[1].set_xlabel('Time (s)')
axes[1].set_ylabel('Vertical velocity (m/s)')
axes[1].set_title('Velocity vs Time')
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3)

axes[2].set_xlabel('Time (s)')
axes[2].set_ylabel('Velocity (m/s)')
axes[2].set_title('Velocity: Simulated (solid) vs Analytical (dashed)')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print expected accelerations
print("Expected vertical accelerations:")
for ratio, label in zip(thrust_ratios, labels):
    T = ratio * T_hover
    a = (2 * T / quad.mass) - quad.g
    print(f"  {label}: a = {a:.3f} m/s^2")

### 3.2 Rotational Response (Differential Thrust)

Differential thrust creates a pitch moment:
$$M = (T_1 - T_2) \cdot L$$

Where $L$ is the arm length. The angular acceleration is:
$$\ddot{\theta} = \frac{M}{I} = \frac{(T_1 - T_2) \cdot L}{I}$$

- More thrust on T1 (right rotor) produces positive moment (nose up)
- More thrust on T2 (left rotor) produces negative moment (nose down)

In [ ]:
# Apply differential thrust while maintaining total thrust at hover
delta_T_values = [-1.0, -0.5, 0.0, 0.5, 1.0]  # N
colors = ['red', 'orange', 'black', 'lightgreen', 'green']
duration_rot = 1.0

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for delta_T, color in zip(delta_T_values, colors):
    # T1 + delta_T, T2 - delta_T (total thrust unchanged)
    T1 = T_hover + delta_T
    T2 = T_hover - delta_T
    control = ConstantControl(jnp.array([T1, T2]))
    
    result = simulate(quad, quad.default_state(), dt=dt, duration=duration_rot, control=control)
    times = np.array(result.times)
    states = np.array(result.states)
    
    label = f'delta_T = {delta_T:+.1f} N'
    axes[0].plot(times, np.degrees(states[:, PQUAD_THETA]), color=color, linewidth=2, label=label)
    axes[1].plot(times, np.degrees(states[:, PQUAD_THETA_DOT]), color=color, linewidth=2, label=label)
    
    # Trajectory
    axes[2].plot(states[:, PQUAD_X], states[:, PQUAD_Z], color=color, linewidth=2, label=label)

axes[0].set_xlabel('Time (s)')
axes[0].set_ylabel('Pitch angle (deg)')
axes[0].set_title('Pitch Angle vs Time')
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.3)

axes[1].set_xlabel('Time (s)')
axes[1].set_ylabel('Pitch rate (deg/s)')
axes[1].set_title('Pitch Rate vs Time')
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3)

axes[2].plot(0, 0, 'ko', markersize=10, label='Start')
axes[2].set_xlabel('x (m)')
axes[2].set_ylabel('z (m)')
axes[2].set_title('Trajectory (tilting causes horizontal motion)')
axes[2].legend(fontsize=8)
axes[2].grid(True, alpha=0.3)
axes[2].axis('equal')

plt.tight_layout()
plt.show()

# Print expected angular accelerations
print("Expected angular accelerations:")
for delta_T in delta_T_values:
    moment = 2 * delta_T * quad.arm_length  # Factor of 2 because delta applied to both rotors
    theta_ddot = moment / quad.inertia_pitch
    print(f"  delta_T = {delta_T:+.1f} N: theta_ddot = {np.degrees(theta_ddot):+.1f} deg/s^2")

### 3.3 Step Response (Thrust Increase from Hover)

Let's examine the response to a step increase in thrust from hover equilibrium.

In [ ]:
# Step response: 20% thrust increase from hover
thrust_increase = 0.20  # 20% increase
T_step = T_hover * (1 + thrust_increase)
control_step = ConstantControl(jnp.array([T_step, T_step]))

duration_step = 3.0
result_step = simulate(quad, quad.default_state(), dt=dt, duration=duration_step, control=control_step)

times_s = np.array(result_step.times)
states_s = np.array(result_step.states)

# Calculate expected values
a_expected = (2 * T_step / quad.mass) - quad.g  # m/s^2
z_expected = 0.5 * a_expected * times_s**2
v_expected = a_expected * times_s

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Height
axes[0].plot(times_s, states_s[:, PQUAD_Z], 'b-', linewidth=2, label='Simulation')
axes[0].plot(times_s, z_expected, 'r--', linewidth=2, label='Analytical')
axes[0].set_xlabel('Time (s)')
axes[0].set_ylabel('Height (m)')
axes[0].set_title(f'Height (20% Thrust Increase)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Velocity
axes[1].plot(times_s, states_s[:, PQUAD_Z_DOT], 'b-', linewidth=2, label='Simulation')
axes[1].plot(times_s, v_expected, 'r--', linewidth=2, label='Analytical')
axes[1].set_xlabel('Time (s)')
axes[1].set_ylabel('Velocity (m/s)')
axes[1].set_title('Vertical Velocity')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Acceleration (from velocity difference)
a_simulated = np.gradient(states_s[:, PQUAD_Z_DOT], times_s)
axes[2].plot(times_s, a_simulated, 'b-', linewidth=2, label='Simulation')
axes[2].axhline(a_expected, color='r', linestyle='--', linewidth=2, label=f'Expected: {a_expected:.3f} m/s^2')
axes[2].set_xlabel('Time (s)')
axes[2].set_ylabel('Acceleration (m/s^2)')
axes[2].set_title('Vertical Acceleration')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Step response analysis:")
print(f"  Thrust per rotor: {T_step:.4f} N (20% above hover)")
print(f"  Expected acceleration: {a_expected:.4f} m/s^2")
print(f"  After {duration_step}s: height = {states_s[-1, PQUAD_Z]:.2f} m, velocity = {states_s[-1, PQUAD_Z_DOT]:.2f} m/s")

---
## 4. Energy and Power Analysis

### Energy Components

The planar quadrotor has three energy components:

**Kinetic energy (translational):**
$$KE_{trans} = \frac{1}{2}m(\dot{x}^2 + \dot{z}^2)$$

**Kinetic energy (rotational):**
$$KE_{rot} = \frac{1}{2}I\dot{\theta}^2$$

**Potential energy:**
$$PE = mgz$$

**Total mechanical energy:**
$$E = KE_{trans} + KE_{rot} + PE$$

In [ ]:
# Demonstrate energy calculation
test_state = quad.create_state(
    z=5.0,           # 5m altitude
    x_dot=2.0,       # 2 m/s horizontal
    z_dot=1.0,       # 1 m/s vertical
    theta_dot=0.5    # 0.5 rad/s pitch rate
)

total_energy = float(quad.energy(test_state))
kinetic_energy = float(quad.kinetic_energy(test_state))
potential_energy = float(quad.potential_energy(test_state))

# Manual calculation for verification
KE_trans_manual = 0.5 * quad.mass * (2.0**2 + 1.0**2)
KE_rot_manual = 0.5 * quad.inertia_pitch * 0.5**2
PE_manual = quad.mass * quad.g * 5.0

print("Energy calculation example:")
print(f"  State: z=5m, x_dot=2m/s, z_dot=1m/s, theta_dot=0.5rad/s")
print(f"\n  Kinetic energy: {kinetic_energy:.4f} J")
print(f"    (translational: {KE_trans_manual:.4f} J, rotational: {KE_rot_manual:.4f} J)")
print(f"  Potential energy: {potential_energy:.4f} J")
print(f"  Total energy: {total_energy:.4f} J")

### Energy Conservation in Freefall

With zero thrust, mechanical energy is conserved. Potential energy converts to kinetic energy as the quadrotor falls.

In [ ]:
# Freefall simulation starting at 10m height
state_freefall = quad.create_state(z=10.0)
result_freefall = simulate(
    quad,
    state_freefall,
    dt=0.0001,  # High resolution for accuracy
    duration=1.4,  # About when it hits ground
    control=ZeroControl(dim=quad.num_controls)
)

times_ff = np.array(result_freefall.times)
states_ff = np.array(result_freefall.states)

# Compute energy at each timestep (vectorized)
energies_total = jax.vmap(quad.energy)(result_freefall.states)
energies_kinetic = jax.vmap(quad.kinetic_energy)(result_freefall.states)
energies_potential = jax.vmap(quad.potential_energy)(result_freefall.states)

energies_total = np.array(energies_total)
energies_kinetic = np.array(energies_kinetic)
energies_potential = np.array(energies_potential)

# Plot
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Energy components
axes[0].plot(times_ff, energies_kinetic, 'r-', linewidth=2, label='Kinetic')
axes[0].plot(times_ff, energies_potential, 'b-', linewidth=2, label='Potential')
axes[0].plot(times_ff, energies_total, 'k-', linewidth=2, label='Total')
axes[0].set_xlabel('Time (s)')
axes[0].set_ylabel('Energy (J)')
axes[0].set_title('Energy Components in Freefall')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Energy conservation error
energy_error = energies_total - energies_total[0]
axes[1].plot(times_ff, energy_error, 'b-', linewidth=2)
axes[1].set_xlabel('Time (s)')
axes[1].set_ylabel('Energy error (J)')
axes[1].set_title('Energy Conservation Error')
axes[1].grid(True, alpha=0.3)

# Height and velocity
ax2_z = axes[2]
ax2_v = ax2_z.twinx()
l1, = ax2_z.plot(times_ff, states_ff[:, PQUAD_Z], 'b-', linewidth=2, label='Height')
l2, = ax2_v.plot(times_ff, states_ff[:, PQUAD_Z_DOT], 'r-', linewidth=2, label='Velocity')
ax2_z.axhline(0, color='k', linestyle='--', alpha=0.3)
ax2_z.set_xlabel('Time (s)')
ax2_z.set_ylabel('Height (m)', color='b')
ax2_v.set_ylabel('Velocity (m/s)', color='r')
axes[2].set_title('Height and Velocity')
axes[2].legend([l1, l2], ['Height', 'Velocity'], loc='upper right')
ax2_z.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Energy conservation in freefall:")
print(f"  Initial energy: {energies_total[0]:.4f} J")
print(f"  Final energy: {energies_total[-1]:.4f} J")
print(f"  Max error: {np.max(np.abs(energy_error)):.2e} J ({100*np.max(np.abs(energy_error))/energies_total[0]:.4f}%)")

### Power Balance

Power is the rate of energy change. For the planar quadrotor:

**Thrust power:**
$$P_{thrust} = \vec{F} \cdot \vec{v} = F_x \dot{x} + F_z \dot{z}$$

Where $F_x = -T\sin(\theta)$ and $F_z = T\cos(\theta)$ are the thrust force components.

**Gravity power:**
$$P_{gravity} = -mg\dot{z}$$

Positive when falling (gravity does work), negative when rising.

In [ ]:
# Power analysis during climb with 30% excess thrust
T_climb = T_hover * 1.3
control_climb = jnp.array([T_climb, T_climb])

result_climb = simulate(
    quad,
    quad.default_state(),
    dt=dt,
    duration=3.0,
    control=ConstantControl(control_climb)
)

times_c = np.array(result_climb.times)
states_c = np.array(result_climb.states)

# Compute power at each timestep
def compute_powers(state, control):
    P_thrust = quad.power_thrust(state, control)
    P_gravity = quad.power_gravity(state)
    return P_thrust, P_gravity

powers_thrust = []
powers_gravity = []
for i in range(len(times_c)):
    P_t, P_g = compute_powers(result_climb.states[i], control_climb)
    powers_thrust.append(float(P_t))
    powers_gravity.append(float(P_g))

powers_thrust = np.array(powers_thrust)
powers_gravity = np.array(powers_gravity)
powers_net = powers_thrust + powers_gravity

# Compute energy change rate from energy time series
energies_climb = np.array(jax.vmap(quad.energy)(result_climb.states))
dE_dt = np.gradient(energies_climb, times_c)

# Plot
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Power components
axes[0].plot(times_c, powers_thrust, 'g-', linewidth=2, label='Thrust power')
axes[0].plot(times_c, powers_gravity, 'r-', linewidth=2, label='Gravity power')
axes[0].plot(times_c, powers_net, 'b-', linewidth=2, label='Net power')
axes[0].axhline(0, color='k', linestyle='--', alpha=0.3)
axes[0].set_xlabel('Time (s)')
axes[0].set_ylabel('Power (W)')
axes[0].set_title('Power Components (30% Thrust Increase)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Power balance verification
axes[1].plot(times_c, powers_net, 'b-', linewidth=2, label='Net power (P_thrust + P_gravity)')
axes[1].plot(times_c, dE_dt, 'r--', linewidth=2, label='dE/dt (from energy)')
axes[1].set_xlabel('Time (s)')
axes[1].set_ylabel('Power (W)')
axes[1].set_title('Power Balance Verification')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Energy vs time
axes[2].plot(times_c, energies_climb, 'b-', linewidth=2)
axes[2].set_xlabel('Time (s)')
axes[2].set_ylabel('Total Energy (J)')
axes[2].set_title('Energy vs Time')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Power analysis (30% excess thrust, climbing):")
print(f"  At t=1s: velocity = {states_c[int(1.0/dt), PQUAD_Z_DOT]:.2f} m/s")
idx = int(1.0 / dt)
print(f"  At t=1s: thrust power = {powers_thrust[idx]:.2f} W")
print(f"  At t=1s: gravity power = {powers_gravity[idx]:.2f} W")
print(f"  At t=1s: net power = {powers_net[idx]:.2f} W")

### Power During Tilted Motion

When the quadrotor is tilted, thrust has both horizontal and vertical components, making the power calculation more interesting.

In [ ]:
# Start tilted at 20 degrees, apply hover thrust
theta_tilt = np.radians(20)
state_tilted = quad.create_state(z=5.0, theta=theta_tilt)

result_tilted = simulate(
    quad,
    state_tilted,
    dt=dt,
    duration=2.0,
    control=ConstantControl(control_hover)
)

times_t = np.array(result_tilted.times)
states_t = np.array(result_tilted.states)

# Compute power
powers_thrust_t = []
powers_gravity_t = []
for i in range(len(times_t)):
    P_t, P_g = compute_powers(result_tilted.states[i], control_hover)
    powers_thrust_t.append(float(P_t))
    powers_gravity_t.append(float(P_g))

powers_thrust_t = np.array(powers_thrust_t)
powers_gravity_t = np.array(powers_gravity_t)
powers_net_t = powers_thrust_t + powers_gravity_t

# Energy
energies_tilted = np.array(jax.vmap(quad.energy)(result_tilted.states))

# Plot
fig, axes = plt.subplots(2, 3, figsize=(14, 8))

# Trajectory
axes[0, 0].plot(states_t[:, PQUAD_X], states_t[:, PQUAD_Z], 'b-', linewidth=2)
axes[0, 0].plot(states_t[0, PQUAD_X], states_t[0, PQUAD_Z], 'go', markersize=10, label='Start')
axes[0, 0].plot(states_t[-1, PQUAD_X], states_t[-1, PQUAD_Z], 'ro', markersize=10, label='End')
axes[0, 0].set_xlabel('x (m)')
axes[0, 0].set_ylabel('z (m)')
axes[0, 0].set_title(f'Trajectory (Initial tilt: {np.degrees(theta_tilt):.0f} deg)')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)
axes[0, 0].axis('equal')

# Velocities
axes[0, 1].plot(times_t, states_t[:, PQUAD_X_DOT], 'b-', linewidth=2, label='x_dot')
axes[0, 1].plot(times_t, states_t[:, PQUAD_Z_DOT], 'r-', linewidth=2, label='z_dot')
axes[0, 1].axhline(0, color='k', linestyle='--', alpha=0.3)
axes[0, 1].set_xlabel('Time (s)')
axes[0, 1].set_ylabel('Velocity (m/s)')
axes[0, 1].set_title('Velocities')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Speed
speeds_t = np.array([float(quad.speed(result_tilted.states[i])) for i in range(len(times_t))])
axes[0, 2].plot(times_t, speeds_t, 'b-', linewidth=2)
axes[0, 2].set_xlabel('Time (s)')
axes[0, 2].set_ylabel('Speed (m/s)')
axes[0, 2].set_title('Speed (velocity magnitude)')
axes[0, 2].grid(True, alpha=0.3)

# Power components
axes[1, 0].plot(times_t, powers_thrust_t, 'g-', linewidth=2, label='Thrust')
axes[1, 0].plot(times_t, powers_gravity_t, 'r-', linewidth=2, label='Gravity')
axes[1, 0].plot(times_t, powers_net_t, 'b-', linewidth=2, label='Net')
axes[1, 0].axhline(0, color='k', linestyle='--', alpha=0.3)
axes[1, 0].set_xlabel('Time (s)')
axes[1, 0].set_ylabel('Power (W)')
axes[1, 0].set_title('Power Components')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Energy components
energies_kinetic_t = np.array(jax.vmap(quad.kinetic_energy)(result_tilted.states))
energies_potential_t = np.array(jax.vmap(quad.potential_energy)(result_tilted.states))
axes[1, 1].plot(times_t, energies_kinetic_t, 'r-', linewidth=2, label='Kinetic')
axes[1, 1].plot(times_t, energies_potential_t, 'b-', linewidth=2, label='Potential')
axes[1, 1].plot(times_t, energies_tilted, 'k-', linewidth=2, label='Total')
axes[1, 1].set_xlabel('Time (s)')
axes[1, 1].set_ylabel('Energy (J)')
axes[1, 1].set_title('Energy Components')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

# Energy change
axes[1, 2].plot(times_t, energies_tilted - energies_tilted[0], 'b-', linewidth=2)
axes[1, 2].set_xlabel('Time (s)')
axes[1, 2].set_ylabel('Energy change (J)')
axes[1, 2].set_title('Energy Change from Initial')
axes[1, 2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Tilted flight analysis:")
print(f"  Initial height: {states_t[0, PQUAD_Z]:.2f} m")
print(f"  Final height: {states_t[-1, PQUAD_Z]:.2f} m (fell {states_t[0, PQUAD_Z] - states_t[-1, PQUAD_Z]:.2f} m)")
print(f"  Final horizontal position: {states_t[-1, PQUAD_X]:.2f} m")
print(f"  Total energy change: {energies_tilted[-1] - energies_tilted[0]:.2f} J")

---

## Bonus: Comparing Parameter Presets

BLUR provides several planar quadrotor presets with different mass and inertia values. Let's compare how they respond to the same differential thrust input.

In [ ]:
from fmd.simulator.params import PLANAR_QUAD_CRAZYFLIE, PLANAR_QUAD_HEAVY

# Compare presets: same differential thrust, different inertia
preset_map = {
    'TEST_DEFAULT': PLANAR_QUAD_TEST_DEFAULT,
    'CRAZYFLIE': PLANAR_QUAD_CRAZYFLIE,
    'HEAVY': PLANAR_QUAD_HEAVY,
}

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for name, params in preset_map.items():
    pq = PlanarQuadrotor(params)
    T_hover = pq.hover_thrust_per_rotor()
    
    # Apply differential thrust (10% of hover thrust)
    delta_T = 0.1 * T_hover
    control_diff = ConstantControl(jnp.array([T_hover + delta_T, T_hover - delta_T]))
    state0_diff = pq.default_state()
    
    result_diff = simulate(pq, state0_diff, dt=0.001, duration=2.0, control=control_diff)
    times_diff = np.array(result_diff.times)
    states_diff = np.array(result_diff.states)
    
    axes[0].plot(times_diff, np.degrees(states_diff[:, PQUAD_THETA]), linewidth=2, label=name)
    axes[1].plot(times_diff, np.degrees(states_diff[:, PQUAD_THETA_DOT]), linewidth=2, label=name)
    
axes[0].set_xlabel('Time (s)')
axes[0].set_ylabel('Pitch angle (deg)')
axes[0].set_title('Pitch Response (10% differential thrust)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].set_xlabel('Time (s)')
axes[1].set_ylabel('Pitch rate (deg/s)')
axes[1].set_title('Pitch Rate')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Summary table
axes[2].axis('off')
text = "Preset Comparison\n" + "=" * 40 + "\n"
text += f"{'Preset':<15} {'Mass (kg)':>10} {'Inertia':>12}\n"
text += "-" * 40 + "\n"
for name, params in preset_map.items():
    pq = PlanarQuadrotor(params)
    text += f"{name:<15} {pq.mass:>10.3f} {pq.inertia_pitch:>12.2e}\n"
axes[2].text(0.0, 1.0, text, va='top', ha='left', family='monospace', fontsize=10)

plt.tight_layout()
plt.show()

print("Key insight: Smaller/lighter quadrotors (like Crazyflie) are much more agile!")
print("They respond faster to the same relative thrust differential.")

## Bonus: Autodiff Through Simulation

JAX enables automatic differentiation through the entire simulation. We can compute gradients of any scalar loss function with respect to initial conditions or model parameters.

In [ ]:
# Autodiff: compute gradient of final position w.r.t. initial tilt
def final_position_loss(theta0):
    """Loss = squared distance from target position after simulation."""
    state0_ad = jnp.array([0.0, 0.0, theta0, 0.0, 0.0, 0.0])
    result_ad = simulate(quad, state0_ad, dt=0.01, duration=1.0, 
                        control=ConstantControl(quad.hover_control()))
    # Minimize squared distance from target (x=1, z=0)
    return (result_ad.states[-1, PQUAD_X] - 1.0)**2 + result_ad.states[-1, PQUAD_Z]**2

# Compute gradient at a test point
theta0_test = 0.1
loss_val = final_position_loss(theta0_test)
grad_val = jax.grad(final_position_loss)(theta0_test)

print("Gradient of loss w.r.t. initial tilt:")
print(f"  theta0 = {np.degrees(theta0_test):.1f} deg")
print(f"  Loss (squared distance from target) = {float(loss_val):.6f}")
print(f"  d(loss)/d(theta0) = {float(grad_val):.6f}")
print(f"\n  Interpretation: Increasing initial tilt by 1 rad changes loss by {float(grad_val):.4f}")

# We can also differentiate w.r.t. model parameters!
def loss_wrt_mass(mass):
    """Loss as a function of mass (how well does fixed thrust work?)."""
    pq = PlanarQuadrotor.from_values(
        mass=mass,
        arm_length=0.25,
        inertia_pitch=0.01,
    )
    # Use thrust designed for 1kg hover
    control_fixed = ConstantControl(jnp.array([4.903, 4.903]))
    state0_mass = jnp.zeros(6)
    result_mass = simulate(pq, state0_mass, dt=0.01, duration=0.5, control=control_fixed)
    return result_mass.states[-1, PQUAD_Z]**2

mass_test = 1.0
loss_mass = loss_wrt_mass(mass_test)
grad_mass = jax.grad(loss_wrt_mass)(mass_test)

print("\nGradient of loss w.r.t. mass:")
print(f"  mass = {mass_test} kg")
print(f"  Loss (final height squared) = {float(loss_mass):.6f}")
print(f"  d(loss)/d(mass) = {float(grad_mass):.6f}")
print("\n  This shows how sensitive performance is to mass uncertainty!")

---
## Summary

This notebook covered the fundamentals of the planar quadrotor model:

**1. System Overview**
- 6 states: position (x, z), pitch (theta), and their derivatives
- 2 controls: thrust from right and left rotors (T1, T2)
- Thrust acts along the body z-axis; differential thrust creates pitch moment

**2. Hover Equilibrium**
- Hover requires T1 = T2 = mg/2 (total thrust equals weight)
- Hover is unstable: small perturbations cause drift without active control
- This instability is why quadrotors need continuous feedback control

**3. Thrust Response**
- Symmetric thrust change produces vertical acceleration: a = T/m - g
- Differential thrust produces angular acceleration: theta_ddot = (T1-T2)*L/I
- Tilting redirects thrust, enabling horizontal motion

**4. Energy and Power**
- Total energy = kinetic (translational + rotational) + potential
- Energy is conserved in freefall (no thrust)
- Thrust power = F dot v; gravity power = -mg*z_dot
- Net power equals rate of energy change

**Next steps:**
- See `docs/control_guide.md` for LQR design, eigenvalue analysis, and timestep selection
- See `docs/simulator_models.md` for the full model reference
